# 00 — CSV setup, stage plans, and storage policy

One row is **one sample**, not one whole study. Four datasets may have different entry stages and output roots.
The CSV declares input state; successful actions are recorded separately. It is never edited automatically.
Start with `config/samples.cells_only.example.csv` for a minimal two-sample migration, or the four-study example.
See `docs/CSV_SCHEMA.md` for all fixed columns and `docs/START_HERE.md` for routes.
This controller environment needs the lightweight package dependencies. Scientific stages use the interpreter paths in settings.

In [ ]:
from pathlib import Path
import os, sys, json
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "vhd" / "control").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter in the extracted pipeline folder (or its notebooks folder).")
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from vhd.control.manifest import load_project, check_policy, save_plan, sample_layout
MANIFEST = Path(os.environ.get("VHD_MANIFEST", ROOT / "config" / "samples.csv"))
SETTINGS = Path(os.environ.get("VHD_SETTINGS", ROOT / "config" / "settings.json"))
if not MANIFEST.exists() or not SETTINGS.exists():
    raise FileNotFoundError("Copy a supplied samples.*.csv to config/samples.csv and settings.g5_24xlarge.example.json to config/settings.json; edit paths and policy first.")
PROJECT = load_project(MANIFEST, SETTINGS)
check_policy(PROJECT)
# Default is a dry run. Set True here only after reviewing the printed plan.
EXECUTE = os.environ.get("VHD_EXECUTE", "0") == "1"
# Optional pilot selection, e.g. ["StudyLegacy__Sample01"]. None selects all applicable rows.
SAMPLE_KEYS = None


## Inspect the execution plan (no biological matrices loaded)

In [ ]:
plan, snapshot = save_plan(PROJECT)
display(plan)
print("Immutable configuration snapshot:", snapshot)
print("EXECUTE:", EXECUTE)

## Inspect active paths and policies

In [ ]:
print(json.dumps(PROJECT["settings"]["storage_policy"], indent=2))
for row in PROJECT["rows"]:
    print(row["sample_key"], row["stage"], "->", row["goal"], row["_plan"])

## Hardware inventory. These are available resources, not promised processing speed.

In [ ]:
from vhd.compute.probe import hardware_inventory
print(json.dumps(hardware_inventory(), indent=2))

## Verify configured scientific interpreters exist. Do not upgrade your working legacy environment.

In [ ]:
for name in ("python_spatial", "python_stardist", "python_scvi", "python_rapids"):
    path = Path(PROJECT["settings"]["execution"][name])
    print(name, "present" if path.is_file() else "EDIT / not available", path)